# D300 Project: Causal ML on Haushofer & Shapiro (2016)

This notebook applies causal machine learning to the GiveDirectly unconditional cash transfer RCT in Kenya. The two main tasks are:
1. Using regularised regression as a specification sensitivity diagnostic to validate the ANCOVA ATE benchmark
2. Estimating heterogeneous treatment effects using an honest causal forest, tested via BLP, GATEs and CLAN

Data from Harvard Dataverse: Haushofer & Shapiro (2016)

Dataverse file can be downloaded from: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/M2GAZN

In [ ]:
import pandas as pd
import numpy as np
import warnings

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as scipy_stats

# ols
import statsmodels.formula.api as smf

# regularised regression (sklearn for cv-lasso, ridge, elastic net)
from sklearn.linear_model import LassoCV, LogisticRegressionCV, ElasticNetCV, RidgeCV
from sklearn import linear_model
from sklearn.preprocessing import StandardScaler

# hdmpy has rlasso (the rigorous lasso with Belloni et al. plug-in penalty)
import hdmpy

# econml for honest causal forest and regression forest (used for BLP nuisance)
from econml.grf import CausalForest, RegressionForest
from econml.policy import PolicyTree

# ensure output folder exists
import os
os.makedirs('output', exist_ok=True)

## 1. Data preparation

The raw dataset has 2,880 rows, two respondents per household (male and female). For household-level outcomes the values are identical across both rows, so we deduplicate by keeping the female respondent row. We also drop pure control villages (we want the within-village comparison only) and 68 attritors with missing endline dates.

In [ ]:
data_raw = pd.read_stata("dataverse_files/UCT_FINAL_CLEAN.dta")

# Keep only treated and spillover households — excludes pure control villages
data_within = data_raw[data_raw['purecontrol'] == 0].copy()

# Deduplicate to household level — keep female respondent row
# household-level outcomes are identical across male/female rows
data_hh = data_within[data_within['femaleres'] == 1].copy()
data_hh = data_hh[data_hh['endlinedate'].notna()].copy()  # drop attritors

# Transfer arm dummies — only meaningful for treated households
# code controls as NaN rather than 0 to avoid contaminating the causal forest
data_hh['femalerec'] = np.where(data_hh['treat'] == 1, data_hh['treatXfemalerec'], np.nan)
data_hh['large'] = np.where(data_hh['treat'] == 1, data_hh['treatXlarge'], np.nan)
data_hh['monthly'] = np.where(data_hh['treat'] == 1, data_hh['treatXmonthly'], np.nan)

# Individual-level dataset — keep both rows for psych wellbeing and female empowerment
# these are measured separately for husband and wife
data_ind = data_within[data_within['endlinedate'].notna()].copy()

print(f"Household sample: n = {len(data_hh)} (treated: {int(data_hh['treat'].sum())}, spillover: {int(data_hh['spillover'].sum())})")
print(f"Individual sample: n = {len(data_ind)}")

# Outcome variables at endline and baseline
outcomes_end = [
    'asset_total_ppp1',
    'cons_nondurable_ppp1',
    'ent_total_rev_ppp1',
    'fs_hhfoodindexnew1',
    'med_hh_healthindex1',
    'ed_index1',
    'psy_index_z1',
    'ih_overall_index_z1'
]

outcomes_base = [
    'asset_total_ppp0',
    'cons_nondurable_ppp0',
    'ent_total_rev_ppp0',
    'fs_hhfoodindexnew0',
    'med_hh_healthindex0',
    'ed_index0',
    'psy_index_z0',
    'ih_overall_index_z0'
]

outcomes_missing = [
    'asset_total_ppp_miss0',
    'cons_nondurable_ppp_miss0',
    'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0',
    'med_hh_healthindex_miss0',
    'ed_index_miss0',
    'psy_index_z_miss0',
    'ih_overall_index_z_miss0'
]

# Baseline covariates used for LASSO/OLS selection
covariate_cols = [
    'b_age', 'b_married', 'b_children', 'b_hhsize', 'b_edu',
    'hh_children0', 'hh_totalmembers0',
    'asset_total_ppp0', 'cons_nondurable_ppp0', 'ent_total_rev_ppp0', 'fs_hhfoodindexnew0',
    'asset_livestock_ppp0', 'asset_durable_ppp0', 'asset_savings_ppp0', 'asset_land_owned_total0',
    'cons_allfood_ppp_m0', 'cons_alcohol_ppp_m0', 'cons_tobacco_ppp_m0',
    'ent_wagelabor0', 'ent_ownfarm0', 'ent_business0',
    'fin_remittances_rec_ppp0',
    'psy_index_z0', 'med_hh_healthindex0', 'ed_index0', 'ih_overall_index_z0',
    'given_mpesa',
    # missing value dummies — baseline coded as 0 when missing, this flags it
    'asset_total_ppp_miss0', 'cons_nondurable_ppp_miss0', 'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0', 'psy_index_z_miss0', 'med_hh_healthindex_miss0',
    'ed_index_miss0', 'ih_overall_index_z_miss0'
]

# Causal forest covariates — treatXfemalerec excluded since it is mechanically correlated with treatment
# I defined to amend if necessary e.g. want to exlcude missing variables '_miss0' etc.
covariate_cols_cf = [
    'b_age', 'b_married', 'b_children', 'b_hhsize', 'b_edu',
    'hh_children0', 'hh_totalmembers0',
    'asset_total_ppp0', 'cons_nondurable_ppp0', 'ent_total_rev_ppp0', 'fs_hhfoodindexnew0',
    'asset_livestock_ppp0', 'asset_durable_ppp0', 'asset_savings_ppp0', 'asset_land_owned_total0',
    'cons_allfood_ppp_m0', 'cons_alcohol_ppp_m0', 'cons_tobacco_ppp_m0',
    'ent_wagelabor0', 'ent_ownfarm0', 'ent_business0',
    'fin_remittances_rec_ppp0',
    'psy_index_z0', 'med_hh_healthindex0', 'ed_index0', 'ih_overall_index_z0',
    'given_mpesa',
    'asset_total_ppp_miss0', 'cons_nondurable_ppp_miss0', 'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0', 'psy_index_z_miss0', 'med_hh_healthindex_miss0',
    'ed_index_miss0', 'ih_overall_index_z_miss0'
]

print(f"\ncovariate_cols: {len(covariate_cols)} variables")
print(f"covariate_cols_cf: {len(covariate_cols_cf)} variables (for causal forest)")

## 2. Randomisation validation: Does treatment predict baseline covariates?

Before doing anything else, we check whether treatment is predictable from baseline characteristics. If it is, we have a randomisation failure and need double-selection (Belloni et al., 2014). If not, the D-on-X step selects nothing and double-selection collapses to single selection.

We run five specifications: logistic regression, CV-LASSO, rlasso (rigorous lasso with plug-in penalty), elastic net, and ridge.

In [ ]:
D = data_hh['treat'].values
village_dummies = pd.get_dummies(data_hh['village'], prefix='village', drop_first=True)
X_rand = pd.concat([data_hh[covariate_cols], village_dummies], axis=1).fillna(0)
col_names = covariate_cols + list(village_dummies.columns)
data_hh_d = pd.concat([data_hh, village_dummies], axis=1).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rand.values)

warnings.filterwarnings('ignore')

# (1) Logistic regression — basic baseline check
formula = 'treat ~ ' + ' + '.join(col_names)
logit_model = smf.logit(formula, data=data_hh_d).fit(method='bfgs', disp=0, maxiter=200)
print(f"Logistic regression: pseudo-R2 = {logit_model.prsquared:.4f}, LR p = {logit_model.llr_pvalue:.3f}")

# (2) CV-LASSO — selects predictors of treatment via cross-validated L1 penalty
lasso_logit = LogisticRegressionCV(Cs=20, cv=5, penalty='l1', solver='liblinear', random_state=42)
lasso_logit.fit(X_scaled, D)
selected_cv = [col_names[i] for i, c in enumerate(lasso_logit.coef_[0]) if c != 0]
print(f"CV-LASSO: {len(selected_cv)} variables selected")

# (3) rlasso — rigorous lasso with Belloni et al. plug-in penalty
# this penalty corrects for multiple testing across p covariates, unlike CV which just minimises MSE
rlasso_d = hdmpy.rlasso(X_rand.values, D, post=True)
selected_idx = rlasso_d.est['index'].values.flatten()
selected_rl = [col_names[i] for i, s in enumerate(selected_idx) if s]
print(f"rlasso: {len(selected_rl)} variables selected")

# (4) Elastic net — L1/L2 mix, CV-tuned
enet_d = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, random_state=42, max_iter=10000)
enet_d.fit(X_scaled, D)
selected_enet = [col_names[i] for i, c in enumerate(enet_d.coef_) if c != 0]
print(f"Elastic Net: {len(selected_enet)} variables selected")

# (5) Ridge — retains all variables, never zeros, so we report R2 instead
ridge_d = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
ridge_d.fit(X_scaled, D)
print(f"Ridge: R2 = {ridge_d.score(X_scaled, D):.4f}, optimal alpha = {ridge_d.alpha_:.1f}")

print("\nConclusion: no method selects any covariates as predictors of treatment.")
print("e_hat(Xi) ≈ 0.5 for all units — randomisation confirmed.")
print("Double selection collapses to single selection throughout.")

warnings.filterwarnings('default')

## 3. Baseline balance

Standard balance check where we regress each baseline outcome on treatment and village fixed effects. Should get p > 0.1 everywhere if randomisation worked.

In [ ]:
# Baseline balance: y_vhib = alpha_v + beta_0 + beta_1*T_vh + eps
# beta_1 = 0 is the null of successful randomisation

results_balance = []

for outcome in outcomes_base:
    model = smf.ols(f'{outcome} ~ treat + C(village)', data=data_hh).fit()
    control_mean = data_hh[data_hh['spillover'] == 1][outcome].mean()
    control_sd = data_hh[data_hh['spillover'] == 1][outcome].std()
    results_balance.append({
        'Outcome': outcome,
        'Control Mean': f"{control_mean:.2f}",
        'Control SD': f"({control_sd:.2f})",
        'Treat Coef': f"{model.params['treat']:.2f}",
        'SE': f"({model.bse['treat']:.2f})",
        'p-value': f"{model.pvalues['treat']:.3f}"
    })

print(pd.DataFrame(results_balance).to_string(index=False))

## 4. ANCOVA benchmark ATE

The ANCOVA specification from Haushofer & Shapiro (2016): endline outcome on treatment, village FE, baseline outcome, and missing dummy. The baseline control reduces residual variance and improves precision without affecting identification.

In [ ]:
# ANCOVA: y_vhiE = alpha_v + beta_1*T_vh + delta_1*y_vhib + delta_2*M_vhiB + eps
# beta_1 is the ATE

results_ate = []

for outcome_end, outcome_base, outcome_miss in zip(outcomes_end, outcomes_base, outcomes_missing):
    model = smf.ols(
        f'{outcome_end} ~ C(village) + treat + {outcome_base} + {outcome_miss}',
        data=data_hh
    ).fit()

    control_mean = data_hh[data_hh['spillover'] == 1][outcome_end].mean()
    control_sd = data_hh[data_hh['spillover'] == 1][outcome_end].std()

    results_ate.append({
        'Outcome': outcome_end,
        'Control Mean': f"{control_mean:.2f}",
        'Control SD': f"({control_sd:.2f})",
        'ATE': f"{model.params['treat']:.2f}",
        'SE': f"({model.bse['treat']:.2f})",
        'p-value': f"{model.pvalues['treat']:.3f}"
    })

print(pd.DataFrame(results_ate).to_string(index=False))

## 5. Post-selection ATE estimation: specification sensitivity diagnostic

The main diagnostic: does the ATE estimate move when we vary the control selector from sparse (rlasso, NX=0-4) to dense (ridge, NX=35)? If the causal parameter is stable across selectors, the ANCOVA benchmark is credible.

For each outcome we run:
- Naive CV-LASSO (treat enters penalised set — biased, shows regularisation shrinkage)
- Post-CV-LASSO OLS (treat forced in, selected controls from above)
- rlasso + post-rlasso OLS
- Post-elastic net OLS
- Post-ridge OLS (all controls retained)

Note: LASSO/ridge/enet run on standardised X since the penalty is scale-sensitive. Post-OLS uses unstandardised for interpretable coefficients.

In [ ]:
# helper functions to strip village dummies and treat from selected variable counts
def count_substantive(selected):
    return sum(1 for c in selected if not c.startswith('village_') and c != 'treat')

def get_substantive(selected):
    return [c for c in selected if not c.startswith('village_') and c != 'treat']

results_all = []

warnings.filterwarnings('ignore')

for outcome_end, outcome_base, outcome_miss in zip(outcomes_end, outcomes_base, outcomes_missing):

    data_clean = data_hh[data_hh[outcome_end].notna()].copy()
    Y = data_clean[outcome_end].values

    village_dummies = pd.get_dummies(data_clean['village'], prefix='village', drop_first=True)
    data_clean_d = pd.concat([data_clean, village_dummies], axis=1)

    # X with treat — for naive lasso (treat enters penalised set)
    X_with_treat = pd.concat([data_clean[['treat'] + covariate_cols], village_dummies], axis=1).fillna(0)
    col_names_with = ['treat'] + covariate_cols + list(village_dummies.columns)

    # X without treat — for ridge and elastic net (we force treat in at post-OLS stage)
    X_controls = pd.concat([data_clean[covariate_cols], village_dummies], axis=1).fillna(0)
    col_names_ctrl = covariate_cols + list(village_dummies.columns)

    scaler1 = StandardScaler()
    X_scaled_with = scaler1.fit_transform(X_with_treat.values)

    scaler2 = StandardScaler()
    X_scaled_ctrl = scaler2.fit_transform(X_controls.values)

    # naive CV-LASSO (treat in penalised set — will be biased toward zero)
    lasso_naive = linear_model.LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso_naive.fit(X_scaled_with, Y)
    selected_naive = [col_names_with[i] for i, c in enumerate(lasso_naive.coef_) if c != 0]
    treat_naive = lasso_naive.coef_[0] / scaler1.scale_[0]  # unscale for reporting
    n_naive = count_substantive(selected_naive)

    # post-CV-LASSO OLS (treat forced in, selected controls from naive lasso)
    ctrl_naive = [c for c in selected_naive if c != 'treat']
    formula_post = outcome_end + ' ~ treat + ' + ' + '.join(ctrl_naive) if ctrl_naive else outcome_end + ' ~ treat'
    post_cv = smf.ols(formula_post, data=data_clean_d).fit()
    n_post_cv = count_substantive(['treat'] + ctrl_naive)

    # rlasso (rigorous lasso — plug-in penalty corrects for multiple testing)
    rl = hdmpy.rlasso(X_with_treat.values, Y, post=True)
    selected_rl_idx = rl.est['index'].values.flatten()
    selected_rl = [col_names_with[i] for i, s in enumerate(selected_rl_idx) if s]
    treat_rl_naive = rl.est['coefficients'].values.flatten()[0]
    n_rl_naive = count_substantive(selected_rl)

    # post-rlasso OLS
    ctrl_rl = [c for c in selected_rl if c != 'treat']
    formula_rl = outcome_end + ' ~ treat + ' + ' + '.join(ctrl_rl) if ctrl_rl else outcome_end + ' ~ treat'
    post_rl = smf.ols(formula_rl, data=data_clean_d).fit()
    n_post_rl = count_substantive(['treat'] + ctrl_rl)

    # elastic net on controls only, then force treat in at post-OLS
    enet = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, random_state=42, max_iter=10000)
    enet.fit(X_scaled_ctrl, Y)
    ctrl_enet = [col_names_ctrl[i] for i, c in enumerate(enet.coef_) if c != 0]
    formula_enet = outcome_end + ' ~ treat + ' + ' + '.join(ctrl_enet) if ctrl_enet else outcome_end + ' ~ treat'
    post_enet = smf.ols(formula_enet, data=data_clean_d).fit()
    n_post_enet = count_substantive(['treat'] + ctrl_enet)

    # ridge on controls only (keeps all, never zeros) then post-OLS with full control set
    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
    ridge.fit(X_scaled_ctrl, Y)
    formula_ridge = outcome_end + ' ~ treat + ' + ' + '.join(col_names_ctrl)
    post_ridge = smf.ols(formula_ridge, data=data_clean_d).fit()
    n_post_ridge = count_substantive(['treat'] + col_names_ctrl)

    print(f"\n{outcome_end}")
    print(f"  CV LASSO selected ({n_naive} vars): {get_substantive(selected_naive)}")
    print(f"  rlasso selected ({n_rl_naive} vars): {get_substantive(selected_rl)}")
    print(f"  Elastic Net selected ({n_post_enet} vars): {ctrl_enet[:5]}..." if len(ctrl_enet) > 5 else f"  Elastic Net selected ({len(ctrl_enet)} vars): {ctrl_enet}")

    results_all.append({
        'Outcome': outcome_end,
        'Naive LASSO': f"{treat_naive:.2f}",
        'Post CV NX': n_naive,
        'Post CV': f"{post_cv.params['treat']:.2f} ({post_cv.bse['treat']:.2f})",
        'rlasso (naive)': f"{treat_rl_naive:.2f}",
        'Post rlasso NX': n_post_rl,
        'Post rlasso': f"{post_rl.params['treat']:.2f} ({post_rl.bse['treat']:.2f})",
        'Post ENet NX': n_post_enet,
        'Post ENet': f"{post_enet.params['treat']:.2f} ({post_enet.bse['treat']:.2f})",
        'Post Ridge NX': n_post_ridge,
        'Post Ridge': f"{post_ridge.params['treat']:.2f} ({post_ridge.bse['treat']:.2f})",
    })

print(pd.DataFrame(results_all).to_string(index=False))

warnings.filterwarnings('default')

## 6. Honest causal forest: estimating individual CATEs

The causal forest (Wager & Athey, 2018) adaptively partitions the covariate space and estimates tau(x) within each leaf as a local difference-in-means. Honesty separates the splitting and estimation samples to prevent overfitting.

We run this for all 8 outcomes. Results stored in cf_results dict for use in BLP/GATE/CLAN below.

In [ ]:
cf_results = {}

for outcome_end, outcome_base, outcome_miss in zip(outcomes_end, outcomes_base, outcomes_missing):

    data_clean = data_hh[data_hh[outcome_end].notna()].copy()
    Y = data_clean[outcome_end].values
    D = data_clean['treat'].values
    X = data_clean[covariate_cols_cf].fillna(0).values

    # honest causal forest — 2000 trees, honest=True separates splitting from estimation
    # random_state for reproducibility
    cf = CausalForest(
        n_estimators=2000,
        min_samples_leaf=5,
        honest=True,
        random_state=42
    )
    cf.fit(X, D, Y)
    tau_hat = cf.predict(X).flatten()

    # get the ANCOVA ATE for comparison on the plot
    ancova = smf.ols(
        f'{outcome_end} ~ C(village) + treat + {outcome_base} + {outcome_miss}',
        data=data_clean
    ).fit()
    ancova_ate = ancova.params['treat']

    # quick sanity check — CF mean CATE should be close to ANCOVA ATE
    print(f"\n{outcome_end}")
    print(f"n = {len(Y)}, ANCOVA ATE = {ancova_ate:.2f}, CF mean CATE = {tau_hat.mean():.2f}")
    print(f"SD of CATEs = {tau_hat.std():.2f}, share positive = {(tau_hat > 0).mean():.3f}")

    # CATE distribution plot — forest vs bootstrap OLS ATE
    # bootstrap the ANCOVA ATE to get a distribution for comparison
    boot_ates = []
    for _ in range(1000):
        idx = np.random.choice(len(data_clean), len(data_clean), replace=True)
        boot_data = data_clean.iloc[idx]
        try:
            b = smf.ols(
                f'{outcome_end} ~ C(village) + treat + {outcome_base} + {outcome_miss}',
                data=boot_data
            ).fit()
            boot_ates.append(b.params['treat'])
        except Exception:
            pass

    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.histogram_bin_edges(np.concatenate([tau_hat, boot_ates]), bins=30)
    ax.hist(boot_ates, bins=bins, alpha=0.6, label='OLS ANCOVA (bootstrap)', colour='orange')
    ax.hist(tau_hat, bins=bins, alpha=0.6, label='Honest Causal Forest CATEs', colour='steelblue')
    ax.set_xlabel('Estimated Treatment Effect')
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of Treatment Effects\n{outcome_end}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'output/cate_dist_{outcome_end}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # store everything for downstream analysis
    cf_results[outcome_end] = {
        'data_clean': data_clean,
        'Y': Y, 'D': D, 'X': X,
        'tau_hat': tau_hat,
        'ancova_ate': ancova_ate
    }

## 7. BLP, GATEs and CLAN

Three-step heterogeneity analysis following Chernozhukov et al. (2018):

**BLP**: Tests whether the forest's CATE rankings are informative. We project Y on (D-p)(tau_hat - tau_bar). A significant beta_2 means units the forest ranks higher actually do better. Note we need the BLP rather than just looking at var(tau_hat) because CATE estimates contain estimation noise — var(tau_hat) overstates true heterogeneity. The BLP instruments this out.

**GATEs**: Sort households into quartiles by predicted CATE and estimate the average effect within each group. Monotonicity from G1 to G4 validates the rankings.

**CLAN**: Compare baseline characteristics between top and bottom quartile to find correlates of high treatment effects.

In [ ]:
# variables to compare in CLAN
clan_vars = [
    'b_age', 'b_married', 'b_children', 'b_hhsize', 'b_edu',
    'asset_total_ppp0', 'cons_nondurable_ppp0', 'fs_hhfoodindexnew0',
    'asset_niceroof0', 'given_mpesa',
    'femalerec', 'large', 'monthly'
]

for outcome, res in cf_results.items():

    data_clean = res['data_clean']
    Y = res['Y']
    D = res['D']
    tau_hat = res['tau_hat']
    ancova_ate = res['ancova_ate']

    print(f"OUTCOME: {outcome}")

    # fit a regression forest on X to estimate E[Y|X] — used as nuisance in BLP
    # this is B(X_i) in the paper, honest to mitigate overfitting
    rf_baseline = RegressionForest(n_estimators=2000, min_samples_leaf=5, honest=True, random_state=42)
    rf_baseline.fit(res['X'], Y)
    B_hat = rf_baseline.predict(res['X']).flatten()

    p_hat = D.mean() # known propensity score = 0.5 by design
    D_tilde = D - p_hat # residualised treatment
    S_hat = tau_hat
    S_bar = S_hat.mean()

    # BLP regression: Y ~ B(X) + beta1*(D-p) + beta2*(D-p)*(tau_hat - tau_bar)
    # beta1 recovers the ATE, beta2 tests for heterogeneity
    blp_df = pd.DataFrame({
        'Y': Y,
        'B_hat': B_hat,
        'D_tilde': D_tilde,
        'DS_tilde': D_tilde * (S_hat - S_bar)
    })
    blp = smf.ols('Y ~ B_hat + D_tilde + DS_tilde - 1', data=blp_df).fit(cov_type='HC1')

    print(f"\nBLP:")
    print(f"  beta_1 (ATE): {blp.params['D_tilde']:.3f} (SE={blp.bse['D_tilde']:.3f}, p={blp.pvalues['D_tilde']:.4f})")
    print(f"  beta_2 (het): {blp.params['DS_tilde']:.3f} (SE={blp.bse['DS_tilde']:.3f}, p={blp.pvalues['DS_tilde']:.4f})")
    if blp.pvalues['DS_tilde'] < 0.05:
        print("  REJECT H0 — significant heterogeneity")
    else:
        print("  FAIL TO REJECT H0")

    # store BLP in results for later
    cf_results[outcome]['blp'] = blp
    cf_results[outcome]['D_tilde'] = D_tilde
    cf_results[outcome]['B_hat'] = B_hat

    # GATEs — sort into quartiles by predicted CATE
    quartiles = pd.qcut(S_hat, q=4, labels=['G1', 'G2', 'G3', 'G4'])
    gate_df = pd.DataFrame({
        'Y': Y, 'B_hat': B_hat,
        'G1D': (quartiles == 'G1').astype(int) * D_tilde,
        'G2D': (quartiles == 'G2').astype(int) * D_tilde,
        'G3D': (quartiles == 'G3').astype(int) * D_tilde,
        'G4D': (quartiles == 'G4').astype(int) * D_tilde,
    })
    gate_model = smf.ols('Y ~ B_hat + G1D + G2D + G3D + G4D - 1', data=gate_df).fit(cov_type='HC1')

    print(f"\nGATEs:")
    for g in ['G1D', 'G2D', 'G3D', 'G4D']:
        coef = gate_model.params[g]
        se = gate_model.bse[g]
        pval = gate_model.pvalues[g]
        ci = gate_model.conf_int().loc[g]
        print(f"  {g.replace('D','')}: {coef:.3f} (SE={se:.3f}, p={pval:.4f}) [{ci[0]:.3f}, {ci[1]:.3f}]")

    # Wald test for G4 - G1
    g4g1 = gate_model.params['G4D'] - gate_model.params['G1D']
    cov = gate_model.cov_params()
    g4g1_se = np.sqrt(cov.loc['G4D','G4D'] + cov.loc['G1D','G1D'] - 2*cov.loc['G4D','G1D'])
    g4g1_p = 2 * (1 - scipy_stats.t.cdf(abs(g4g1/g4g1_se), df=len(Y)-5))
    print(f"  G4-G1 spread: {g4g1:.3f} (SE={g4g1_se:.3f}, p={g4g1_p:.4f})")

    cf_results[outcome]['gate_model'] = gate_model
    cf_results[outcome]['quartiles'] = quartiles

    # CLAN — compare G1 vs G4 on baseline characteristics using Welch t-tests
    data_clean['quartile'] = np.array(quartiles)
    g1 = data_clean[data_clean['quartile'] == 'G1']
    g4 = data_clean[data_clean['quartile'] == 'G4']
    print(f"\nCLAN (n G1={len(g1)}, n G4={len(g4)}):")
    print(f"  {'Variable':<30} {'G1':>8} {'G4':>8} {'Diff':>8} {'d':>6} {'p':>8}")

    for var in clan_vars:
        if var not in data_clean.columns:
            continue
        v1 = g1[var].dropna()
        v4 = g4[var].dropna()
        if len(v1) == 0 or len(v4) == 0:
            continue
        diff = v4.mean() - v1.mean()
        tstat, pval = scipy_stats.ttest_ind(v4, v1, equal_var=False)
        # Cohen's d using pooled SD
        pooled_sd = np.sqrt(((len(v1)-1)*v1.std()**2 + (len(v4)-1)*v4.std()**2) / (len(v1)+len(v4)-2))
        d = diff / pooled_sd if pooled_sd > 0 else np.nan
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
        print(f"  {var:<30} {v1.mean():>8.3f} {v4.mean():>8.3f} {diff:>8.3f} {d:>6.3f} {pval:>8.4f} {sig}")

    # GATE plot
    gates = [gate_model.params[f'G{k}D'] for k in range(1, 5)]
    gate_se = [gate_model.bse[f'G{k}D'] for k in range(1, 5)]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(range(1, 5), gates, yerr=[1.96*s for s in gate_se], capsize=5,
           colour='steelblue', alpha=0.7, edgecolour='white')
    ax.axhline(tau_hat.mean(), colour='red', linestyle='--', linewidth=1.5,
               label=f'CF mean CATE = {tau_hat.mean():.2f}')
    ax.axhline(ancova_ate, colour='black', linestyle=':', linewidth=1.5,
               label=f'OLS ANCOVA = {ancova_ate:.2f}')
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(['G1\n(Lowest)', 'G2', 'G3', 'G4\n(Highest)'])
    ax.set_ylabel('GATE Estimate')
    ax.set_title(f'Group Average Treatment Effects\n{outcome}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'output/gates_{outcome}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. OLS of tau_hat on X: projected heterogeneity

This is a descriptive exercise to regress estimated CATEs on baseline covariates to see which dimensions the forest leans on most. Coefficients have no causal interpretation since tau_hat is a generated regressor, but it helps identify which variables drive heterogeneity across outcomes.

(Also not included in D300 report).

In [ ]:
extra_vars = ['femalerec', 'large', 'monthly']

for outcome, res in cf_results.items():

    data_clean = res['data_clean']
    tau_hat = res['tau_hat']

    reg_df = data_clean[covariate_cols].copy().fillna(0)
    reg_df['tau_hat'] = tau_hat
    # add transfer arm dummies — fill NaN with 0 for controls
    for v in extra_vars:
        reg_df[v] = data_clean[v].fillna(0).values

    formula = 'tau_hat ~ ' + ' + '.join(covariate_cols + extra_vars)
    cate_ols = smf.ols(formula, data=reg_df).fit(cov_type='HC1')

    # only print variables significant at 10% to keep output manageable
    results_ols = pd.DataFrame({
        'coef': cate_ols.params,
        'se': cate_ols.bse,
        'pvalue': cate_ols.pvalues
    }).drop('Intercept').sort_values('pvalue')

    sig_results = results_ols[results_ols['pvalue'] < 0.10]

    print(f"\n{outcome}")
    print(f"  {'Variable':<35} {'Coef':>8} {'SE':>8} {'p':>8}")
    for var, row in sig_results.iterrows():
        sig = '***' if row['pvalue'] < 0.01 else '**' if row['pvalue'] < 0.05 else '*'
        print(f"  {var:<35} {row['coef']:>8.3f} {row['se']:>8.3f} {row['pvalue']:>8.4f} {sig}")
    if len(sig_results) == 0:
        print("No predictors significant at 10%")

## 9. Policy trees

Policy trees (Athey & Wager, 2021) translate the forest's CATE estimates into simple decision rules. The tree picks splits on observable baseline characteristics to assign treatment, maximising the expected treatment effect among the assigned group.

Two trees:
- **Assets**: almost all households have positive predicted CATE so an unconstrained tree treats everyone. We impose a 50% budget constraint by centring the reward around the median tau which forces the tree to find the households that benefit most relative to the median.
- **Health**: predicted effects span positive and negative so no constraint is needed since the tree naturally separates households.

This section is an interesting next step for policy application but is not included in the report (due to the scope of the analysis).

In [ ]:
# drop missing value dummies from policy tree features — not useful for targeting decisions
missing_cols = [
    'asset_total_ppp_miss0', 'cons_nondurable_ppp_miss0', 'ent_total_rev_ppp_miss0',
    'fs_hhfoodindexnew_miss0', 'psy_index_z_miss0', 'med_hh_healthindex_miss0',
    'ed_index_miss0', 'ih_overall_index_z_miss0'
]
covariate_cols_pt = [c for c in covariate_cols_cf if c not in missing_cols]

# cleaner variable names for the tree plot
clean_names = {
    'b_age': 'Age of HH head', 'b_edu': 'Education (yrs)', 'b_hhsize': 'HH size',
    'b_married': 'Married', 'b_children': 'No. children',
    'asset_total_ppp0': 'Total assets (PPP)', 'cons_nondurable_ppp0': 'Nondurable cons. (PPP)',
    'asset_livestock_ppp0': 'Livestock (PPP)', 'asset_durable_ppp0': 'Durable assets (PPP)',
    'asset_savings_ppp0': 'Savings (PPP)', 'asset_land_owned_total0': 'Land owned',
    'cons_allfood_ppp_m0': 'Food cons. (PPP)', 'cons_alcohol_ppp_m0': 'Alcohol cons.',
    'cons_tobacco_ppp_m0': 'Tobacco cons.', 'ent_wagelabor0': 'Wage labour',
    'ent_ownfarm0': 'Own farm', 'ent_business0': 'Business owner',
    'fin_remittances_rec_ppp0': 'Remittances (PPP)', 'psy_index_z0': 'Psych wellbeing',
    'med_hh_healthindex0': 'Health index', 'ed_index0': 'Education index',
    'ih_overall_index_z0': 'Female empowerment', 'given_mpesa': 'Has M-Pesa',
    'hh_children0': 'Children (survey)', 'hh_totalmembers0': 'HH members'
}
feature_names_pt = [clean_names.get(c, c) for c in covariate_cols_pt]

def plot_tree(pt, title, fname):
    treat_patch = mpatches.Patch(colour='blue', label='Treat')
    notreat_patch = mpatches.Patch(colour='orange', label='Do not treat')
    fig, ax = plt.subplots(figsize=(20, 9))
    pt.plot(feature_names=feature_names_pt, ax=ax, fontsize=9)
    ax.set_title(f'Policy Tree: {title}', fontsize=12, pad=12)
    ax.legend(handles=[treat_patch, notreat_patch], loc='lower center',
              bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=9, frameon=False)
    plt.tight_layout()
    plt.savefig(f'output/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

# Tree 1: Assets — 50% budget constraint
# centre the reward around median tau so tree identifies top 50% not just 'positive effect'
res_a = cf_results['asset_total_ppp1']
tau_a = res_a['tau_hat']
X_a = res_a['data_clean'][covariate_cols_pt].fillna(0).values

median_a = np.median(tau_a)
reward_a = np.column_stack([np.zeros(len(tau_a)), tau_a - median_a])

pt_assets = PolicyTree(max_depth=3, min_samples_leaf=20, random_state=42)
pt_assets.fit(X_a, reward_a)
mask_a = pt_assets.predict(X_a) == 1

print(f"Policy Tree: Assets (50% budget, median tau = {median_a:.0f} PPP)")
print(f"Targeted: {mask_a.sum()} ({mask_a.mean():.1%})")
print(f"Mean tau targeted: {tau_a[mask_a].mean():.2f} PPP")
print(f"Mean tau untargeted: {tau_a[~mask_a].mean():.2f} PPP")
print(f"Welfare gain vs random: {tau_a[mask_a].mean() - tau_a.mean():.2f} PPP per HH")
plot_tree(pt_assets, f'Non-land Assets (top 50% by predicted CATE)', 'policy_tree_assets.png')

# Tree 2: Health — no budget constraint needed, effects span positive and negative
res_h = cf_results['med_hh_healthindex1']
tau_h = res_h['tau_hat']
X_h = res_h['data_clean'][covariate_cols_pt].fillna(0).values

reward_h = np.column_stack([np.zeros(len(tau_h)), tau_h])

pt_health = PolicyTree(max_depth=3, min_samples_leaf=20, random_state=42)
pt_health.fit(X_h, reward_h)

# apply minimum effect threshold — only target leaves where predicted gain >= 0.03 SD
MIN_EFFECT = 0.03
leaf_ids = pt_health.apply(X_h)
leaf_mean = {lid: tau_h[leaf_ids == lid].mean() for lid in np.unique(leaf_ids)}
unit_leaf_mean = np.array([leaf_mean[lid] for lid in leaf_ids])
mask_h = (pt_health.predict(X_h) == 1) & (unit_leaf_mean >= MIN_EFFECT)

print(f"\nPolicy Tree: Health (min {MIN_EFFECT} SD improvement)")
print(f"Targeted: {mask_h.sum()} ({mask_h.mean():.1%})")
if mask_h.sum() > 0:
    print(f"Mean tau targeted: {tau_h[mask_h].mean():.4f} SD")
    print(f"Mean tau untargeted: {tau_h[~mask_h].mean():.4f} SD")
    print(f"Welfare gain vs random: {tau_h[mask_h].mean() - tau_h.mean():.4f} SD per HH")
plot_tree(pt_health, 'Health Index (positive predicted effect)', 'policy_tree_health.png')

# How much overlap between the two targeting rules?
both = mask_a & mask_h
print(f"\nOverlap between targeting rules:")
print(f" Assets only: {(mask_a & ~mask_h).sum()}")
print(f" Health only: {(mask_h & ~mask_a).sum()}")
print(f" Both: {both.sum()} ({both.sum()/mask_h.sum():.1%} of health-targeted)")